# Epsilon Fund — Strategy Testing
---

In [2]:
import pandas as pd
import numpy as np
import sys

# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research'  # ← change to yours
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + '/infrastructure/data')
sys.path.append(ROOT + '/infrastructure/backtester')

from binance_client import get_binance_client, get_data, get_multiple_data
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: `365` (1y) · `730` (2y) · `1825` (5y) · `2555` (7y, recommended minimum)

In [3]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1d'
LOOKBACK = 2555

# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)

print(f'{SYMBOL} | {INTERVAL} | {df.index[0].date()} -> {df.index[-1].date()} | {len(df)} rows')
df.tail(3)

BTCUSDT | 1d | 2019-03-23 -> 2026-03-20 | 2555 rows


,Open,High,Low,Close,Volume
Time,,,,,
2026-03-18,73909.37,74672.34,70500.00,71246.54,23392.88093
2026-03-19,71246.55,71613.79,68793.35,69930.00,22364.81819
2026-03-20,69930.01,71367.00,69837.10,70313.98,9609.30962


---
## Strategy

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> Signals are shifted 1 bar by the engine — no need to shift manually.

In [25]:
strategy_df = df.copy()

# ── Strategy logic ─────────────────────────────────────────────────────────────

# --- ATR (Wilder's smoothing, matches Pine Script RMA) ---
def calculate_atr(df, period):
    high       = df['High']
    low        = df['Low']
    prev_close = df['Close'].shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, min_periods=period, adjust=False).mean()


# --- EMA TRIGGER ---
strategy_df['EMA_9']  = strategy_df['Close'].ewm(span=9,  adjust=False).mean()
strategy_df['EMA_21'] = strategy_df['Close'].ewm(span=21, adjust=False).mean()

strategy_df['ema_cross_up']   = (
    (strategy_df['EMA_9'] > strategy_df['EMA_21']) &
    (strategy_df['EMA_9'].shift(1) <= strategy_df['EMA_21'].shift(1))
)
strategy_df['ema_cross_down'] = (
    (strategy_df['EMA_9'] < strategy_df['EMA_21']) &
    (strategy_df['EMA_9'].shift(1) >= strategy_df['EMA_21'].shift(1))
)


# --- SUPERTREND ---
atr_period = 14
factor     = 1

atr = calculate_atr(strategy_df, atr_period)
hl2 = (strategy_df['High'] + strategy_df['Low']) / 2

raw_upper = hl2 + factor * atr
raw_lower = hl2 - factor * atr

n          = len(strategy_df)
upper      = np.zeros(n)
lower      = np.zeros(n)
direction  = np.zeros(n)   # -1 = bullish, +1 = bearish

first_valid_i = atr.first_valid_index()
start_i       = strategy_df.index.get_loc(first_valid_i)

for i in range(start_i, n):
    if i == start_i:
        upper[i]     = raw_upper.iloc[i]
        lower[i]     = raw_lower.iloc[i]
        direction[i] = 1
        continue

    close_prev = strategy_df['Close'].iloc[i - 1]
    close_curr = strategy_df['Close'].iloc[i]

    upper[i] = raw_upper.iloc[i] if (raw_upper.iloc[i] < upper[i-1] or close_prev > upper[i-1]) \
               else upper[i-1]
    lower[i] = raw_lower.iloc[i] if (raw_lower.iloc[i] > lower[i-1] or close_prev < lower[i-1]) \
               else lower[i-1]

    if direction[i-1] == 1:
        direction[i] = -1 if close_curr > upper[i-1] else 1
    else:
        direction[i] =  1 if close_curr < lower[i-1] else -1

strategy_df['supertrend'] = np.where(direction == -1,
                                     lower,   # bullish: lower band is the support line
                                     upper)   # bearish: upper band is the resistance line
strategy_df['direction']  = direction

# --- PULLBACK ZONE ---
atr_len      = 10
pullback_atr = calculate_atr(strategy_df, atr_len)

strategy_df['long_in_range'] = (
    (strategy_df['direction'] == -1) &
    (strategy_df['Close'] >= strategy_df['supertrend']) &
    (strategy_df['Close'] <= strategy_df['supertrend'] + 2*pullback_atr)
)
strategy_df['short_in_range'] = (
    (strategy_df['direction'] == 1) &
    (strategy_df['Close'] <= strategy_df['supertrend']) &
    (strategy_df['Close'] >= strategy_df['supertrend'] - 1*pullback_atr)
)

# ← add these two lines here
strategy_df['long_zone_recent']  = strategy_df['long_in_range'].rolling(5).max().astype(bool)
strategy_df['short_zone_recent'] = strategy_df['short_in_range'].rolling(5).max().astype(bool)

# --- EXIT LEVELS ---
strategy_df['long_exit_level']  = (strategy_df['Low'].shift(1)  + strategy_df['Low'].shift(2))  / 2
strategy_df['short_exit_level'] = (strategy_df['High'].shift(1) + strategy_df['High'].shift(2)) / 2


# --- POSITION LOOP ---
strategy_df['position'] = 0
in_pos = 0

for i in range(2, len(strategy_df)):   # start at 2: exit levels need 2 prior bars
    idx      = strategy_df.index[i]
    curr     = strategy_df.iloc[i]
    prev     = strategy_df.iloc[i - 1]

    # Skip warmup bars where ATR/supertrend not yet valid
    if pd.isna(curr['supertrend']):
        continue

    if in_pos == 1:   # in long
        defensive     = curr['Close'] < curr['supertrend']
        trailing_exit = curr['Close'] < curr['long_exit_level']
        if defensive or trailing_exit:
            in_pos = 0

    elif in_pos == -1:   # in short
        defensive     = curr['Close'] > curr['supertrend']
        trailing_exit = curr['Close'] > curr['short_exit_level']
        if defensive or trailing_exit:
            in_pos = 0

    if in_pos == 0:
        if curr['long_zone_recent'] and curr['ema_cross_up']:
            in_pos = 1
        elif curr['short_zone_recent'] and curr['ema_cross_down']:
            in_pos = -1
    strategy_df.loc[idx, 'position'] = in_pos

# ──────────────────────────────────────────────────────────────────────────────

# ──────────────────────────────────────────────────────────────────────────────

first_valid = strategy_df[['supertrend', 'long_exit_level']].first_valid_index()
strategy_df = strategy_df.loc[first_valid:]

strategy_df['position'] = strategy_df['position'].fillna(0).astype(int)

strategy_df['position'].value_counts()

position
 0    2059
 1     309
-1     187
Name: count, dtype: int64

---
## Backtest

| Parameter | Options | Default |
|---|---|---|
| `cost` | Cost per trade as decimal — `0.001` = 0.1% | `0.0` |
| `show_plot` | `True` / `False` | `True` |
| `save_html` | Filename string or `None` | `None` |
| `show_trades` | Overlay entry/exit markers on price chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | same asset |

In [26]:
results = backtest(
    data           = strategy_df,
    cost           = 0.0,
    show_plot      = True,
    save_html      = None,
    show_trades    = False,
    benchmark_data = None
)

print(f"Return        {results['total_return']*100:>8.2f}%")
print(f"Sharpe        {results['sharpe_ratio']:>8.2f}")
print(f"Max Drawdown  {results['max_drawdown']*100:>8.2f}%")
print(f"Profit Factor {results['profit_factor']:>8.2f}")
print(f"Win Rate      {results['win_rate']*100:>8.2f}%")
print(f"# Trades      {results['num_trades']:>8}")

Return          144.39%
Sharpe            0.62
Max Drawdown    -37.31%
Profit Factor     1.76
Win Rate         46.34%
# Trades            82
